# AI Engineering Interview Prep
## Section: Resume-Based Questions

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 651+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (30 Qs)",
    "18. FastAPI (30 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "21. System Design & Scenario-Based (100 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 📄 Section 19 — Resume-Based Questions

### Q1. Tell me about Nyaya-Pro. Why did you choose FAISS instead of a managed vector database like Pinecone?
**A:** I built Nyaya-Pro to solve a real problem: navigating the transition to the new Indian penal codes (BNS, BNSS). I chose FAISS for two reasons. First, data residency and privacy: legal queries are highly sensitive, and keeping the document embeddings on our local disk/server container rather than sending them to a third-party managed service was a major compliance win. Second, cost and scale: the entire corpus of the Constitution, BNS, BNSS, and CPC is relatively small (under 150,000 words). Loading it into a local FAISS index takes less than 20MB of VRAM/RAM. Using Pinecone would have introduced unnecessary network latency, subscription costs, and setup complexity for a dataset that easily fits in memory.

### Q2. What was the role of the agentic query router in Nyaya-Pro? How does it work?
**A:** In a standard RAG pipeline, the query goes straight to vector search. But Indian law has distinct domains (procedural law vs. penal code vs. constitutional issues). If a user asks a question about bail, searching the entire corpus might pull irrelevant constitutional clauses. I built a router using Gemini to classify the query first (e.g., "Criminal Procedure", "Penal Code", "Constitutional"). Based on this classification, the system filters the metadata in FAISS to search *only* the relevant document sets. It keeps the context clean, prevents noise, and ensures that the top retrieved chunks are actually on-topic.

### Q3. Why did you use a semantic cross-encoder reranker in Nyaya-Pro? How did it affect latency?
**A:** Bi-encoders (like the sentence-transformer used for indexing) are great for retrieving a broad pool of candidates fast, but they don't capture the deep semantic relationship between the query and the text. Because a wrong citation in a legal context is a critical failure, I added a cross-encoder reranker. It takes the query and the top 10 retrieved chunks, feeds them together into a joint attention model, and re-scores them. This raised our faithfulness metric to 94% on RAGAS evals. It did add about 200-250ms of latency, but in legal research, accuracy is non-negotiable. I hid the latency in the UI by streaming the initial thoughts/metadata while the final ranking and generation completed.

### Q4. Explain your multi-agent CrewAI pipeline in JobPilot AI. How do the agents collaborate?
**A:** JobPilot AI is a pipeline that takes a user's resume and a job description and outputs an ATS-optimized resume and cover letter. Instead of one massive prompt, I split the work between three specialized agents:
1. **The Resume Analyzer:** Extracts the candidate's core strengths and identifies gaps relative to the job description.
2. **The ATS Optimizer:** Suggests rephrasings, matches keywords, and aligns bullet points without fabricating experience.
3. **The Cover Letter Writer:** Synthesizes the analysis into a professional, human-toned cover letter.
They pass their outputs sequentially. The analyzer's report goes to the optimizer, and the optimizer's draft goes to the writer. This separation of concerns prevented the LLM from getting "lost" or diluting its instructions, leading to much cleaner outputs.

### Q5. In JobPilot AI, why did you use Ollama locally for development but Groq API in production?
**A:** This was a pure developer productivity and cost decision. During development, I was running dozens of test iterations, editing agent prompts, and testing edge cases. Running that via cloud APIs would have cost money and hit rate limits. With Ollama, I ran Llama 3 locally for free, without worrying about APIs or network latency. But local models run slow on standard development machines. For production, I needed sub-second response times for a good user experience. I switched to Groq's API serving Llama-3.3-70b. Groq's LPU hardware delivers over 400 tokens per second, giving production users near-instant responses at a very low cost.

### Q6. How does ExamGenie AI parse study PDFs and generate interactive MCQs? How do you prevent hallucinations?
**A:** ExamGenie AI uses Angular 18 on the front end and calls Gemini 2.5 Flash. Gemini has a massive context window and native multimodal capabilities, so instead of using messy text extraction libraries that break on tables and layouts, I pass the PDF document pages directly to the model. To prevent hallucinations (like generating MCQs with incorrect keys or wrong explanations), I do two things:
1. **Strict schema enforcement:** I use structured JSON outputs matching a Pydantic-like typescript interface.
2. **Self-Consistency grounding:** The prompt instructs the model to first locate the exact sentence in the PDF that supports the question, write it down as a `grounding_source` field, and only then generate the MCQ. If the model can't find a direct quote, it won't generate the question.

### Q7. You worked at Yotta Infrastructures as a Senior Software Developer. How did you optimize the CCTV live-playback streams?
**A:** At Yotta, our internal operations dashboard had to display real-time CCTV feeds for datacenter server rooms. The initial implementation had high latency and stuttering because it was pulling raw video streams directly. I optimized this by implementing HLS (HTTP Live Streaming) chunking and setting up a media-proxy cache. We adjusted the chunk size to 2-second segments to balance loading speed with buffering, and offloaded the rendering logic from the main thread using Web Workers. This reduced stream start latency by over 40% and stopped UI blocking, providing a smooth, real-time monitoring experience for the datacenter operations team.

### Q8. How did you improve dashboard performance by 30% using NgRx in OneYotta Portal?
**A:** The OneYotta Portal handles massive amounts of data-center resource data, and the UI was laggy due to redundant API calls and unnecessary component re-renders. Every time a user switched tabs, the app was refetching resources. I refactored the state management to NgRx. By creating a single source of truth and using memoized selectors (`createSelector`), we ensured components only re-rendered when their specific slice of state actually changed. I also implemented an optimistic update pattern and custom caching in the NgRx effects. This cut API traffic significantly and made the application feel instantaneous, improving overall dashboard responsiveness by roughly 30%.

### Q9. How did you handle authentication in OneYotta Portal using Keycloak and Azure AD?
**A:** We needed a secure, single sign-on (SSO) solution for thousands of enterprise users. We used Keycloak as our identity broker. I integrated Keycloak with the Angular frontend using the `keycloak-angular` library. For enterprise clients, we configured Keycloak to federate identity to their Azure Active Directory (Azure AD) via SAML/OIDC. On the frontend, I set up route guards to intercept unauthorized requests, redirect users to the Keycloak login page, and handle the OAuth 2.0 authorization code flow. Once authenticated, the bearer token was appended to outgoing API requests using an HTTP Interceptor, with automatic token-refresh handled in the background via RxJS streams.

### Q10. How do you balance your strong frontend background (Angular, Flutter) with your transition into AI Engineering?
**A:** I don't see them as separate; I see them as a massive competitive advantage. Most AI engineers are great at Jupyter Notebooks but don't know how to build a production-grade UI, handle state, or build a clean mobile app. Because of my experience at Yotta, I can write the FastAPI backend, optimize the RAG pipeline, and then build a polished Angular 18 web app (like ExamGenie) or a Next.js frontend (like Nyaya-Pro) to actually deliver that AI to the user. An AI model is only as good as the interface the user interacts with, and my background allows me to build the entire product end-to-end.